## Импорты

In [ ]:
import pandas as pd
import numpy as np
from itertools import product
np.random.seed(42)

Базовые распределения

In [ ]:
banks_dist = {
    'Сбер': 0.25, 'Альфа-банк': 0.15, 'ВТБ': 0.15, 'Т-Банк': 0.12,
    'Газпромбанк': 0.10, 'МКБ': 0.08, 'Совкомбанк': 0.05,
    'Почта банк': 0.05, 'Дом.РФ': 0.05
}
banks = list(banks_dist.keys())

categories = ['Разработка', 'Аналитика', 'Администрирование', 'Тестирование', 'Дизайн']
experience_levels = ['Нет опыта', 'от 1 до 3 лет', '3-6 лет', 'Свыше 6 лет']
cities = ['Москва', 'Санкт-Петербург', 'Нижний Новгород', 'Казань', 'Екатеринбург',
          'Самара', 'Уфа', 'Омск', 'Волгоград', 'Другой']

salary_match_levels = [
    'от 80000 до 149 999 руб.', 'Нет данных', 'Нет совпадений',
    'от 150000 до 249999 руб', 'До 79 999 руб', 'от 250000 до 399999 руб.', 'более 400000руб'
]

age_groups = ['до 25', '25-35', '36-45', '46-55', '55+']
genders = ['Мужской', 'Женский']
education_levels = ['Высшее бакалавр', 'Магистратура', 'Только курсы', 'Среднее', 'Нет данных']

Функции зависимостей

In [ ]:
def get_age_by_experience(exp):
    age_map = {
        'Нет опыта': ['до 25'],
        'от 1 до 3 лет': ['до 25', '25-35'],
        '3-6 лет': ['25-35', '36-45'],
        'Свыше 6 лет': ['36-45', '46-55', '55+']
    }
    probs = {'Нет опыта': [1.0], 'от 1 до 3 лет': [0.4, 0.6],
             '3-6 лет': [0.7, 0.3], 'Свыше 6 лет': [0.5, 0.4, 0.1]}
    return np.random.choice(age_map[exp], p=probs[exp])

def get_education_by_category(cat):
    if cat == 'Разработка': return np.random.choice(education_levels, p=[0.6, 0.3, 0.0, 0.0, 0.1])
    elif cat == 'Аналитика': return np.random.choice(education_levels, p=[0.4, 0.5, 0.0, 0.0, 0.1])
    elif cat == 'Дизайн': return np.random.choice(education_levels, p=[0.3, 0.0, 0.5, 0.15, 0.05])
    else: return np.random.choice(education_levels, p=[0.5, 0.2, 0.1, 0.1, 0.1])

def get_university(city):
    if city == 'Москва': return np.random.choice(['МГУ', 'ВШЭ', 'Бауманка', 'Нет'], p=[0.3,0.3,0.2,0.2])
    elif city == 'Санкт-Петербург': return np.random.choice(['СПбГУ', 'ИТМО', 'Нет'], p=[0.4,0.3,0.3])
    else: return np.random.choice(['Региональный', 'Нет'], p=[0.4, 0.6])

Генерация полного грида резюме

In [ ]:
all_combinations = list(product(categories, experience_levels, salary_match_levels,
                               age_groups, cities, education_levels, genders))

resume_groups = []
for cat, exp, sal_match, age, city, edu, gender in all_combinations:
    if np.random.random() > 0.05:
        freshness = np.random.choice(['день назад', 'неделя назад', 'месяц назад',
                                     'полгода назад', 'год назад'], p=[0.3,0.25,0.2,0.15,0.1])
        uni = get_university(city)
        status = np.random.choice(['-', 'Проф. курсы'], p=[0.75, 0.25])

        resume_count = max(20, int(np.random.poisson(100)))
        empty_desc_pct = np.random.uniform(0.05, 0.10)
        empty_desc_count = int(resume_count * empty_desc_pct)

        avg_sal_res = np.random.normal(160000, 40000)
        median_sal_res = int(avg_sal_res * 0.75)

        last_job_exp = {'Нет опыта': 0.5, 'от 1 до 3 лет': 2.0,
                       '3-6 лет': 4.5, 'Свыше 6 лет': 8.0}[exp]

        resume_groups.append({
            'Категория сотрудника': cat,
            'Совпадение опыта': exp,
            'Совпадение зарплатных ожиданий': sal_match,
            'Возраст': age,
            'Свежесть обновления резюме': freshness,
            'Уровень образования': edu,
            'Пол': gender,
            'Город': city,
            'Университет': uni,
            'Количество резюме': resume_count,
            'Количество резюме, где поменялось место работы': int(resume_count * 0.45),
            'Среднее время перехода': f"{int(np.random.poisson(16))} мес",
            'Статус': status,
            'Медианная зарплата из резюме': median_sal_res,
            'Средняя зарплата по резюме': int(avg_sal_res),
            'Опыт на последнем месте работы, лет': round(np.random.normal(last_job_exp, 1), 1),
            'Процент кандидатов без описания данных': round(empty_desc_pct * 100, 1)
        })

resume_df = pd.DataFrame(resume_groups)
resume_agg = resume_df.groupby([
    'Категория сотрудника', 'Совпадение опыта', 'Совпадение зарплатных ожиданий',
    'Возраст', 'Уровень образования', 'Пол', 'Город', 'Университет'
]).agg({
    'Количество резюме': 'sum',
    'Количество резюме, где поменялось место работы': 'sum',
    'Медианная зарплата из резюме': 'median',
    'Средняя зарплата по резюме': 'mean',
    'Опыт на последнем месте работы, лет': 'mean',
    'Процент кандидатов без описания данных': 'mean',
    'Свежесть обновления резюме': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'месяц назад',
    'Среднее время перехода': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else '16 мес',
    'Статус': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else '-'
}).round({'Медианная зарплата из резюме': 0, 'Средняя зарплата по резюме': 0,
          'Опыт на последнем месте работы, лет': 1}).reset_index()

Генерация датасета вакансий

In [ ]:
vac_groups = []
for cat, exp, sal_match, age, city, edu, _ in product(categories, experience_levels, salary_match_levels,
                                                     age_groups, cities, education_levels, ['']):
    if np.random.random() > 0.2:
        bank = np.random.choice(banks, p=list(banks_dist.values()))
        vac_count = max(5, int(np.random.poisson(30 * banks_dist[bank] * 2)))

        salary_map = {
            'До 79 999 руб': 60000, 'от 80000 до 149 999 руб.': 120000,
            'от 150000 до 249999 руб': 200000, 'от 250000 до 399999 руб.': 320000,
            'более 400000руб': 450000
        }
        avg_sal = salary_map.get(sal_match, 200000) + np.random.normal(0, 20000)
        median_sal = int(avg_sal * 0.75)

        vac_groups.append({
            'Банк': bank,
            'Категория сотрудника': cat,
            'Совпадение опыта': exp,
            'Совпадение зарплатных ожиданий': sal_match,
            'Возраст': age,
            'Город': city,
            'Количество вакансий': vac_count,
            'Медианная зарплата из вакансий': median_sal,
            'Средняя зарплата по вакансиям': int(avg_sal),
            'Уровень образования': edu
        })

vac_df = pd.DataFrame(vac_groups)
vac_agg = vac_df.groupby([
    'Банк', 'Категория сотрудника', 'Совпадение опыта', 'Совпадение зарплатных ожиданий',
    'Возраст', 'Город', 'Уровень образования'
]).agg({
    'Количество вакансий': 'sum',
    'Медианная зарплата из вакансий': 'median',
    'Средняя зарплата по вакансиям': 'mean'
}).round(0).reset_index()

Финальный джойн и сохранение

In [ ]:
join_keys_vac = ['Категория сотрудника', 'Совпадение опыта', 'Совпадение зарплатных ожиданий',
                'Возраст', 'Город', 'Уровень образования']
join_keys_res = ['Категория сотрудника', 'Совпадение опыта', 'Совпадение зарплатных ожиданий',
                'Возраст', 'Уровень образования', 'Город', 'Пол', 'Университет']

final_df = pd.merge(vac_agg, resume_agg, on=join_keys_vac, how='inner', suffixes=('_vac', '_res'))
final_df['Индекс конкуренции'] = (final_df['Количество резюме'] / final_df['Количество вакансий']).round(2)

final_df.to_csv('FINAL_COMPLETE_DATASET.csv', index=False)
vac_agg.to_csv('vacancies_complete.csv', index=False)
resume_agg.to_csv('resumes_complete.csv', index=False)